# Gap recovery study (coarse hourly and full five-minute)

This notebook calls the same library runner and loads the same published artifacts as
`market-data research-run gap_recovery` / `gap_recovery_opening`. It contains no second
execution or publication path.

Limits stated up front (D-012): direct hourly bars begin at 10:00 New York time, so the
coarse study cannot see 09:30-09:59; the five-minute study can, from the 09:30 bar's close
at 09:35. Intraday volume is IEX-only and is not composite liquidity; the studies measure
zero-volume bar shares and hourly/5-minute close agreement on the actual cohort rather than
assuming fidelity. Test-period events are observed but never summarized here.

In [ ]:
import polars as pl

from marketdata.config import load_config
from marketdata.query import load_research_observations
from marketdata.research import run_registered_event_study
from marketdata.store.meta import MetaStore
from marketdata.studies.gap_recovery import DEFAULT_PARAMETERS, STUDY_NAME

config = load_config()
DEFAULT_PARAMETERS

## Run (or reuse) a published run

Each call publishes a new immutable run. Set `RUN_ID` to reuse an existing one.

In [ ]:
RUN_ID = None  # e.g. "3f2a..." to reuse a published run

if RUN_ID is None:
    published = run_registered_event_study(config, STUDY_NAME, {})
    RUN_ID = published.run_id
    print(published)
RUN_ID

## Coverage funnel and metrics

The runner's `event_audit.*` metrics give candidates → eligible → selected → evaluable / missing.

In [ ]:
with MetaStore(config.meta_path) as meta:
    metrics = pl.DataFrame([dict(row) for row in meta.research_metrics(RUN_ID)])
funnel = metrics.filter(pl.col("metric_name").str.starts_with("event_audit."))
funnel.select("metric_name", "value", "dimensions_json")

In [ ]:
summary = metrics.filter(
    ~pl.col("metric_name").str.starts_with("event_audit.")
    & pl.col("metric_name").is_in(
        ["evaluable", "median_return", "hit_rate_target", "mean_excess_return"]
    )
)
summary.sort("dimensions_json", "metric_name")

## Observations

One row per selected event and checkpoint. `measured_return` is raw hourly close over raw EOD open;
`gap_return` is adjusted open over adjusted prior close. Corporate-action days are flagged and their
raw-basis recovered fraction is null.

In [ ]:
observations = load_research_observations(config, run_ids=[RUN_ID])
(
    observations.filter(pl.col("period") != "test")
    .group_by("period", "observation_label")
    .agg(
        pl.len().alias("events"),
        (pl.col("outcome_status") == "evaluable").sum().alias("evaluable"),
        pl.col("measured_return").median().alias("median_return"),
        pl.col("excess_return").median().alias("median_excess"),
        pl.col("reached_target_at_checkpoint").mean().alias("hit_rate_1pct"),
    )
    .sort("period", "observation_label")
)

## Slices by liquidity and year (development + validation only)

In [ ]:
(
    observations.filter(
        (pl.col("period") != "test") & (pl.col("observation_label") == "session_close")
    )
    .with_columns(
        pl.col("event_date").dt.year().alias("year"),
        pl.when(pl.col("adv_dollars") >= 1e9)
        .then(pl.lit("adv>=1B"))
        .when(pl.col("adv_dollars") >= 2e8)
        .then(pl.lit("adv 200M-1B"))
        .otherwise(pl.lit("adv<200M"))
        .alias("liquidity"),
    )
    .group_by("year", "liquidity")
    .agg(
        pl.len().alias("events"),
        pl.col("measured_return").median().alias("median_close_return"),
        pl.col("reached_target_at_checkpoint").mean().alias("hit_rate_1pct"),
    )
    .sort("year", "liquidity")
)

## Full five-minute opening-window study

Same selection as the coarse study plus a five-minute IEX density screen; outcomes are
session-relative five-minute bar closes from 09:35 onward. Two return bases: from the raw
EOD opening print (observed, not an assumed fill) and from the 09:30 bar's close (the first
completed price a morning decision could act on).

In [ ]:
from marketdata.studies.gap_recovery_opening import STUDY_NAME as OPENING_STUDY

OPENING_RUN_ID = None  # set to reuse a published opening-window run

if OPENING_RUN_ID is None:
    opening = run_registered_event_study(config, OPENING_STUDY, {})
    OPENING_RUN_ID = opening.run_id
    print(opening)
OPENING_RUN_ID

In [ ]:
opening_obs = load_research_observations(config, run_ids=[OPENING_RUN_ID])
(
    opening_obs.filter(pl.col("period") != "test")
    .group_by("period", "observation_label")
    .agg(
        pl.len().alias("events"),
        (pl.col("outcome_status") == "evaluable").sum().alias("evaluable"),
        pl.col("measured_return").median().alias("median_from_open"),
        pl.col("measured_return_from_first_bar").median().alias("median_from_0935"),
        pl.col("reached_target_at_checkpoint").mean().alias("hit_rate_1pct"),
        pl.col("excess_return").mean().alias("mean_excess"),
        (
            pl.col("zero_volume_bars_through_checkpoint").sum()
            / pl.col("bars_through_checkpoint").sum()
        ).alias("zero_volume_share"),
        pl.col("hourly_close_matches_5min").mean().alias("hourly_agreement"),
    )
    .sort("period", "observation_label")
)

### Slices: stock vs ETF, market regime, market trend

Regime and trend come from SPY's prior-window realized volatility and return (decision-time
inputs only); asset type comes from alias metadata.

In [ ]:
close_label = "16:00_close_of_15:55_bar"
for column in ("asset_type", "market_regime", "market_trend"):
    display(
        opening_obs.filter(
            (pl.col("period") != "test") & (pl.col("observation_label") == close_label)
        )
        .group_by("period", column)
        .agg(
            pl.len().alias("events"),
            pl.col("measured_return").median().alias("median_close_return"),
            pl.col("reached_target_at_checkpoint").mean().alias("hit_rate_1pct"),
        )
        .sort("period", column)
    )

## Coarse versus full over common events

`compare_gap_studies` pairs each common event at the availability times both studies share
(11:00 through 16:00) and summarizes the checkpoints only the five-minute study can see.

In [ ]:
from marketdata.studies.compare import compare_gap_studies

comparison = compare_gap_studies(
    config, coarse_run_id=RUN_ID, opening_run_id=OPENING_RUN_ID
)
print(comparison.coarse_events, comparison.opening_events, comparison.common_events)
display(comparison.checkpoint_agreement)
comparison.opening_window